# ResNet-50-D encoder, trained from scratch

Tests the design in `docs/cnn_encoder_architecture.md`. The encoder and SGD are defined in this notebook;
data, training loop, evaluation and Grad-CAM all come from `src/`, which this notebook does not change.

| Run | Encoder | Recipe | Input | Question it answers | Runs by default |
|---|---|---|---|---|---|
| R0 | ResNet-18 | Old | 224 | Baseline: 61.9% val, from the earlier Run B | Already done |
| R1 | ResNet-18 | New | 224 | How much does the recipe alone add? | No (later session) |
| **R2** | **ResNet-50-D** | **New** | **224** | **How much does the architecture add?** | **Yes** |
| R3 | SE-ResNet-50-D | New | 224 | Does attention help? | No |
| R4 | ResNet-50-D | New | 320 | Does extra resolution help? | No |

R5 (CutMix/Mixup) from the design doc is not here: it needs changes to the training loop, and the doc only
adds it if we get that far.

**The default plan fits in 3 hours**
- It tunes the learning rate (2 trials × 20 epochs), then trains R2 for up to 150 epochs.
- R2's epoch count comes from the time left, using the speed measured during tuning, so a slower GPU means fewer
  epochs rather than a skipped run. If R2 gets fewer than 150, the output says so.
- R2 is not compared with R1 yet. To run R1 later, see the comment above `RUNS_TO_RUN` in section 1.

**Where to run**
- **Kaggle**: enable *GPU T4* and *Internet*, add the secret `GITHUB_TOKEN`, then *Save & Run All*.
- **Colab or VS Code with a Colab kernel**: connect to a T4, and either add the Colab secret `GITHUB_TOKEN` or
  upload `.env`. Sessions can end early, so lower `MAX_SESSION_HOURS` to suit.
- **Without a GPU**: `QUICK_CHECK` turns on automatically and runs every step on a few batches. Accuracy from a
  quick check means nothing; it only shows the code works.

**Across sessions:** Kaggle and Colab start empty. To skip tuning next time, copy the printed settings into
`BEST_PARAMS`.

## 0. Setup

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
os.environ["TQDM_DISABLE"] = "1"     # keeps saved notebook output readable

ON_KAGGLE = os.path.exists("/kaggle/working")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False


def load_env():
    # KEY=value lines from an uploaded .env; values already in the environment win
    folders = (Path.cwd(), Path("/content"), Path("/content/drive/MyDrive"))
    env_file = next((f / ".env" for f in folders if (f / ".env").exists()), None)
    if env_file is None:
        return
    print("Secrets loaded from", env_file)
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("'\""))


def get_secret(name):
    # From .env, then Kaggle or Colab secrets; None when it is not set
    if name in os.environ:
        return os.environ[name]
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if ON_KAGGLE or ON_COLAB:
    load_env()
    GITHUB_TOKEN = get_secret("GITHUB_TOKEN")     # only needed while the repo is private
    credentials = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""

    REPO = "omarsaher/image-classfication-and-caption"
    BRANCH = "feature/scratch-cnn"
    WORKDIR = "/tmp/repo" if ON_KAGGLE else "/content/repo"                    # code + data
    OUTPUTS = "/kaggle/working/outputs" if ON_KAGGLE else "/content/outputs"   # results

    # The token is removed from the remote URL right after cloning
    if not os.path.exists(WORKDIR):
        result = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH,
             f"https://{credentials}github.com/{REPO}.git", WORKDIR],
            capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        )
        stderr = result.stderr.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else result.stderr
        print(stderr[-2000:])
        if result.returncode != 0:
            raise RuntimeError(f"Could not clone branch '{BRANCH}'. While the repo is private, "
                               "check that GITHUB_TOKEN can access it.")
        subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)
    os.environ.pop("GITHUB_TOKEN", None)          # training does not need it
    os.makedirs(OUTPUTS, exist_ok=True)
    if not os.path.exists(f"{WORKDIR}/outputs"):
        os.symlink(OUTPUTS, f"{WORKDIR}/outputs")
    os.chdir(WORKDIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)                   # project paths are relative to the repo root

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import torch

print("repo root:", Path.cwd())
print("torch", torch.__version__, "| device:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Settings

Everything you are likely to change is here.

In [ ]:
QUICK_CHECK = not torch.cuda.is_available()   # no GPU: run each step on a few batches only

# Later session for R1: use ["R1"], paste BEST_PARAMS, and set EPOCHS to the number R2 actually ran.
# R3 and R4 are also available, but each needs its own session.
RUNS_TO_RUN = ["R2"]
RESNET18_WIDTH = 0.5            # for R1, must match Run B: see its config.yaml (0.5 or 1.0)

# Hyperparameter search. Each encoder listed here gets its own search.
# R1 reuses R2's settings by default, which saves hours but slightly favours R2.
# If R2 only wins narrowly, add "resnet18" and run again for a fairer comparison.
TUNE_ENCODERS = ["resnet50d"]
TUNE_GRID = {"train.lr": [0.003, 0.0075], "train.weight_decay": [0.0005]}
TUNE_EPOCHS = 20
TRY_BATCH_16 = False            # one extra trial of the best settings; off to save time

# To skip tuning, paste the settings printed at the end of an earlier session, e.g.
# {"train.lr": 0.003, "train.weight_decay": 0.0005, "data.batch_size": 32}
BEST_PARAMS = None

EPOCHS = 150                    # the most a run will train; fewer if the time left requires it
WARMUP_EPOCHS = 5
PATIENCE = 40

RUN_TEST_EVAL = True            # evaluates only the winner, chosen on validation
TIE_MARGIN = 0.03               # a bigger model must beat a simpler one by this much (see section 7)
MAX_SESSION_HOURS = 3.0         # the whole notebook, including setup and evaluation
RESERVE_HOURS = 0.2             # kept free for the test set, Grad-CAM and the zip
MIN_EPOCHS = 30                 # below this a run is not worth starting

if QUICK_CHECK:
    EPOCHS, WARMUP_EPOCHS, TUNE_EPOCHS = 1, 0, 1
    TUNE_GRID = {"train.lr": [0.003], "train.weight_decay": [0.0005]}

print("mode:", "QUICK CHECK (accuracy means nothing)" if QUICK_CHECK else "full training")
print("runs:", RUNS_TO_RUN, "| tuning:", "skipped" if BEST_PARAMS else TUNE_ENCODERS)

## 2. The encoder

Follows the drawings in section 3 of the design doc. It implements the project's `Encoder` interface, so
registering it is enough for `src/`'s classifier, checkpoints, evaluation and Grad-CAM to use it.

A checkpoint from this encoder can only be loaded where `ResNet50D` is registered, so the `src.cli` commands will not
find it until the class moves into `src/`.

In [ ]:
from torch import nn

from src.models.encoders.base import Encoder
from src.models.encoders.scratch_resnet import BASE_WIDTHS, _scaled, conv_bn_relu
from src.models.registry import register_encoder

EXPANSION = 4


class DropPath(nn.Module):
    # Stochastic depth: while training, skip a block's residual for a random subset of images
    def __init__(self, p=0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty(x.size(0), 1, 1, 1).bernoulli_(keep)
        return x * mask / keep


class SqueezeExcite(nn.Module):
    # Channel attention: reweights channels using a summary of the whole feature map
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.gate(x)


class BottleneckD(nn.Module):
    # 1x1 -> 3x3 -> 1x1. The stride sits on the 3x3 conv, and the shortcut downsamples with
    # average pooling, so neither path skips input positions.
    def __init__(self, in_ch, mid_ch, stride=1, drop_path=0.0, use_se=False):
        super().__init__()
        out_ch = mid_ch * EXPANSION
        self.conv1 = nn.Conv2d(in_ch, mid_ch, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.conv2 = nn.Conv2d(mid_ch, mid_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_ch)
        self.conv3 = nn.Conv2d(mid_ch, out_ch, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.se = SqueezeExcite(out_ch) if use_se else nn.Identity()
        self.drop_path = DropPath(drop_path)

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            # ceil_mode matches the 3x3 conv's rounding, so odd-sized feature maps also line up
            layers = [nn.AvgPool2d(stride, stride, ceil_mode=True, count_include_pad=False)] if stride != 1 else []
            layers += [nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
            self.shortcut = nn.Sequential(*layers)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.drop_path(self.se(out))
        return self.relu(out + self.shortcut(x))


class ResNet50D(Encoder):
    def __init__(self, width_mult=1.0, blocks_per_stage=(3, 4, 6, 3), drop_path=0.1,
                 use_se=False, in_channels=3):
        super().__init__()
        widths = [_scaled(c, width_mult) for c in BASE_WIDTHS]
        stem_mid = _scaled(32, width_mult)

        self.stem = nn.Sequential(
            conv_bn_relu(in_channels, stem_mid, stride=2),
            conv_bn_relu(stem_mid, stem_mid),
            conv_bn_relu(stem_mid, widths[0]),
        )
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)

        # Drop probability grows linearly with depth, from 0 in the first block to drop_path in the last
        total_blocks = sum(blocks_per_stage)
        block_index = 0
        stages = []
        in_ch = widths[0]
        for i, (mid_ch, n_blocks) in enumerate(zip(widths, blocks_per_stage)):
            blocks = []
            for j in range(n_blocks):
                stride = 2 if i > 0 and j == 0 else 1
                p = drop_path * block_index / max(1, total_blocks - 1)
                blocks.append(BottleneckD(in_ch, mid_ch, stride, p, use_se))
                in_ch = mid_ch * EXPANSION
                block_index += 1
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.ModuleList(stages)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_channels = [widths[0]] + [w * EXPANSION for w in widths]
        self.feature_dim = widths[-1] * EXPANSION
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
        # Every residual block starts as an identity mapping
        for m in self.modules():
            if isinstance(m, BottleneckD):
                nn.init.zeros_(m.bn3.weight)

    @property
    def gradcam_layer(self):
        return self.stages[-1]

    def forward_features(self, x):
        features = {"c1": self.stem(x)}
        x = self.maxpool(features["c1"])
        for i, stage in enumerate(self.stages, start=2):
            x = stage(x)
            features[f"c{i}"] = x
        features["pooled"] = self.pool(x).flatten(1)
        return features


register_encoder("scratch_resnet50d", ResNet50D)

## 3. Check the model against the design

These checks stop the notebook before any training if the model doesn't match the design doc.

In [ ]:
from src.models.registry import build_classifier, count_parameters

R50D_MODEL = {"name": "scratch_resnet50d", "num_classes": 200, "dropout": 0.3, "width_mult": 1.0,
              "blocks_per_stage": [3, 4, 6, 3], "drop_path": 0.1, "use_se": False}

model = build_classifier(R50D_MODEL)
n_params = count_parameters(model)
assert abs(n_params - 23.94e6) < 0.01e6, n_params
se_params = count_parameters(build_classifier({**R50D_MODEL, "use_se": True}))
assert abs(se_params - 26.47e6) < 0.01e6, se_params
print(f"ResNet-50-D: {n_params:,} parameters | with SE: {se_params:,}")

model.eval()
with torch.no_grad():
    for size in (224, 320):
        features = model.encoder.forward_features(torch.zeros(1, 3, size, size))
        print(f"input {size}:", {k: tuple(v.shape[1:]) for k, v in features.items()})
        for name, channels, stride in zip(("c1", "c2", "c3", "c4", "c5"),
                                          model.encoder.out_channels, (2, 4, 8, 16, 32)):
            assert features[name].shape[1:] == (channels, size // stride, size // stride), name
assert features["pooled"].shape[1] == model.encoder.feature_dim == 2048

# Sizes that reach an odd feature map must work too: the shortcut and main path have to round alike
with torch.no_grad():
    for size in (112, 225, 256, 333):
        model.encoder.forward_features(torch.zeros(1, 3, size, size))

# Colorization needs a 1-channel input
gray = ResNet50D(in_channels=1).eval()
with torch.no_grad():
    assert gray.forward_features(torch.zeros(1, 1, 224, 224))["c5"].shape[1:] == (2048, 7, 7)

# One backward pass: every parameter gets a gradient, and the first loss is close to ln(200) = 5.30
model.train()
loss = nn.functional.cross_entropy(model(torch.randn(4, 3, 224, 224)), torch.tensor([0, 1, 2, 3]))
loss.backward()
assert all(p.grad is not None for p in model.parameters())
print(f"loss at initialisation: {loss.item():.2f}")

probabilities = [round(b.drop_path.p, 3) for stage in model.encoder.stages for b in stage]
assert probabilities[0] == 0.0 and probabilities[-1] == 0.1 and len(probabilities) == 16
print("all checks passed")
del model, gray

## 4. Recipe and run definitions

The project's `Trainer` only knows AdamW, so this cell replaces its optimizer function with SGD for this session.
Warmup, cosine decay, label smoothing, mixed precision, early stopping and checkpoints all stay as they are.

In [ ]:
import importlib.util
import json
import statistics

from src.training import trainer as trainer_module
from src.training.train_classifier import train_classifier
from src.utils.config import load_config


def build_sgd(model, lr, weight_decay):
    # Same parameter groups as the project's AdamW: no decay on biases or BatchNorm
    decay, no_decay = [], []
    for param in model.parameters():
        if param.requires_grad:
            (no_decay if param.ndim <= 1 else decay).append(param)
    return torch.optim.SGD(
        [{"params": decay, "weight_decay": weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr, momentum=0.9, nesterov=True,
    )


trainer_module.build_optimizer = build_sgd

workers = 0 if not torch.cuda.is_available() else min(4, os.cpu_count() or 1)
has_mlflow = importlib.util.find_spec("mlflow") is not None
tracking = "mlflow" if has_mlflow and not QUICK_CHECK else "none"

COMMON = [
    "data.use_offline_augmented=true",    # originals + copies, the best data so far (Run B)
    "data.batch_size=32",
    f"data.num_workers={workers}",
    "train.optimizer=sgd_nesterov",       # recorded in config.yaml only; build_sgd does the work
    f"train.epochs={EPOCHS}",
    f"train.warmup_epochs={WARMUP_EPOCHS}",
    f"train.early_stopping_patience={PATIENCE}",
    f"tracking.backend={tracking}",
]
if QUICK_CHECK:
    COMMON += ["train.max_batches=2", "data.batch_size=8"]

ENCODER_SETTINGS = {
    "resnet18": ["model.name=scratch_resnet", f"model.width_mult={RESNET18_WIDTH}",
                 "model.blocks_per_stage=[2,2,2,2]"],
    "resnet50d": ["model.name=scratch_resnet50d", "model.width_mult=1.0",
                  "model.blocks_per_stage=[3,4,6,3]", "model.drop_path=0.1", "model.use_se=false"],
    "se_resnet50d": ["model.name=scratch_resnet50d", "model.width_mult=1.0",
                     "model.blocks_per_stage=[3,4,6,3]", "model.drop_path=0.1", "model.use_se=true"],
}

# Listed from simplest to most complex, which is the order section 7 compares them in.
# time_factor scales the time estimate: 320 px costs about twice the compute of 224 px.
RUNS = {
    "R1": {"name": "R1_resnet18_new_recipe", "encoder": "resnet18", "extra": [], "time_factor": 1.0},
    "R2": {"name": "R2_resnet50d", "encoder": "resnet50d", "extra": [], "time_factor": 1.0},
    "R3": {"name": "R3_se_resnet50d", "encoder": "se_resnet50d", "extra": [], "time_factor": 1.0},
    "R4": {"name": "R4_resnet50d_320", "encoder": "resnet50d", "time_factor": 2.0,
           "extra": ["data.image_size=320", "data.resize_size=366"]},
}
SUFFIX = "_smoke" if QUICK_CHECK else ""


def make_cfg(name, encoder, params, extra=(), output_dir="outputs/classification"):
    overrides = [
        *COMMON, *ENCODER_SETTINGS[encoder], *[f"{k}={v}" for k, v in params.items()], *extra,
        f"experiment.name={name}{SUFFIX}", f"experiment.output_dir={output_dir}",
    ]
    return load_config("configs/cls_scratch.yaml", overrides)


epoch_seconds = None        # measured from the latest finished run


def hours_used():
    return (time.time() - SESSION_START) / 3600


def epochs_that_fit(time_factor=1.0):
    # How many epochs fit in the time left, up to EPOCHS. Needs a speed measured by an earlier run.
    if not epoch_seconds:
        return EPOCHS
    seconds_left = (MAX_SESSION_HOURS - RESERVE_HOURS - hours_used()) * 3600
    return min(EPOCHS, int(seconds_left / (epoch_seconds * time_factor)))


def run_training(cfg, time_factor=1.0):
    # Train one model, unless it would not fit in the session; returns None when skipped
    global epoch_seconds
    name = cfg["experiment"]["name"]
    if epoch_seconds:
        needed = cfg["train"]["epochs"] * epoch_seconds * time_factor / 3600
        print(f"[{name}] estimated {needed:.1f} h | {hours_used():.1f} h used so far")
        if hours_used() + needed > MAX_SESSION_HOURS:
            print(f"[{name}] SKIPPED: would pass MAX_SESSION_HOURS = {MAX_SESSION_HOURS}")
            return None
    result = train_classifier(cfg)
    epoch_seconds = statistics.median(row["epoch_time_sec"] for row in result.history) / time_factor
    print(f"[{name}] {result.elapsed_sec / 60:.1f} min | {epoch_seconds:.0f} s per epoch")
    return result


probe = build_sgd(nn.Linear(2, 2), 0.01, 0.001)
print("optimizer:", type(probe).__name__, "| nesterov:", probe.defaults["nesterov"],
      "| data loader workers:", workers, "| tracking:", tracking)

## 5. Tune the learning rate

To fit the 3-hour plan, the search compares two learning rates at batch size 32, for 20 epochs each. Short trials are
a rough guide for SGD, so treat a small gap between them as a tie. An earlier run suggests 0.001 learns too slowly, so
it is left out. Finished trials are saved after each one, so rerunning this cell in the same session continues where
it stopped.

In [ ]:
import itertools


def tune(encoder):
    output_dir = Path("outputs") / f"tuning_{encoder}{SUFFIX}"
    output_dir.mkdir(parents=True, exist_ok=True)
    results_path = output_dir / "results.json"
    results = json.loads(results_path.read_text(encoding="utf-8")) if results_path.exists() else []

    def run_trial(params):
        if any(row["params"] == params for row in results):
            return True
        cfg = make_cfg(f"trial_{len(results):02d}", encoder, params, extra=[
            f"train.epochs={TUNE_EPOCHS}", "train.warmup_epochs=1",
            f"train.early_stopping_patience={TUNE_EPOCHS}", "train.save_checkpoint=false",
            f"tracking.group=tune_{encoder}",
        ], output_dir=str(output_dir))
        print(f"\n[{encoder}] {params}")
        result = run_training(cfg)
        if result is None:
            return False
        results.append({"params": params, "best_val_acc": result.best_val_acc,
                        "best_epoch": result.best_epoch, "time_sec": result.elapsed_sec})
        results_path.write_text(json.dumps(results, indent=1), encoding="utf-8")
        return True

    keys = list(TUNE_GRID)
    for values in itertools.product(*TUNE_GRID.values()):
        if not run_trial(dict(zip(keys, values))):
            break
    if not results:
        raise RuntimeError(f"No tuning trial for {encoder} fits in the session")

    best = max(results, key=lambda row: row["best_val_acc"])
    if TRY_BATCH_16 and "data.batch_size" not in best["params"]:
        run_trial({**best["params"], "data.batch_size": 16})
        best = max(results, key=lambda row: row["best_val_acc"])

    print(f"\n{encoder} trials:")
    for row in sorted(results, key=lambda row: -row["best_val_acc"]):
        print(f"  val {row['best_val_acc']:.3f}  epoch {row['best_epoch']:3d}  {row['params']}")
    return {"data.batch_size": 32, **best["params"]}


tuned = {encoder: BEST_PARAMS for encoder in ENCODER_SETTINGS} if BEST_PARAMS else {
    encoder: tune(encoder) for encoder in TUNE_ENCODERS
}
print("\nSettings to paste into BEST_PARAMS next session:", json.dumps(tuned))

## 6. Train

Every run uses the same split, seed and recipe. The epoch count is set before training starts, so the learning rate
schedule still completes its full cosine decay. Each run writes `best.pt`, `history.json`, `curves.png` and
`config.yaml` to its own folder under `outputs/classification/`.

In [ ]:
def params_for(encoder):
    # An encoder without its own search reuses the ResNet-50-D settings
    if encoder in tuned:
        return tuned[encoder]
    return tuned["resnet50d"]


results = {}
for run in RUNS_TO_RUN:
    spec = RUNS[run]
    params = params_for(spec["encoder"])
    epochs = epochs_that_fit(spec["time_factor"])
    print(f"\n===== {run}: {spec['name']} | {params} | {epochs} epochs")
    if epochs < min(MIN_EPOCHS, EPOCHS):
        print(f"[{spec['name']}] SKIPPED: only {epochs} epochs fit in the time left")
        continue
    if epochs < EPOCHS:
        print(f"Time limit: running {epochs} of {EPOCHS} epochs. Use the same number for runs compared with this one.")
    cfg = make_cfg(spec["name"], spec["encoder"], params, extra=[*spec["extra"], f"train.epochs={epochs}"])
    result = run_training(cfg, spec["time_factor"])
    if result is not None:
        results[run] = result
print(f"\nFinished {list(results)} after {hours_used():.1f} h")

## 7. Compare the runs and choose a winner

The winner is chosen on validation only, starting from the simplest model. A more complex model replaces the
current winner only if it beats it by at least `TIE_MARGIN`. With 1,000 validation images, one accuracy has a standard
error of about 1.5 points, so a smaller gap between two single runs is treated as a tie.

In [ ]:
from IPython.display import Image as ShowImage, display

from src.visualization.plots import plot_run_comparison

OUTPUT_ROOT = Path("outputs/classification")
R0_VAL = 0.619      # Run B from reports/figures/task1_2/runs_A_B_curves.png

summaries, histories = {}, {}
for run, spec in RUNS.items():
    run_dir = OUTPUT_ROOT / f"{spec['name']}{SUFFIX}"
    if run not in results or not (run_dir / "best.pt").exists():
        continue
    history = json.loads((run_dir / "history.json").read_text(encoding="utf-8"))
    best = max(history, key=lambda row: row["val_acc"])
    config = load_config(run_dir / "config.yaml")
    summaries[run] = {
        "dir": run_dir, "name": spec["name"], "val": best["val_acc"], "train": best["train_acc"],
        "epoch": best["epoch"], "epochs_run": len(history),
        "hours": sum(row["epoch_time_sec"] for row in history) / 3600,
        "params": count_parameters(build_classifier(config["model"])),
    }
    histories[f"{run}: {spec['name']}"] = history

print(f"{'run':<4}{'model':<26}{'params':>9}{'val':>8}{'train':>8}{'gap':>8}{'best at':>9}{'hours':>7}")
print(f"{'R0':<4}{'ResNet-18, old recipe':<26}{'':>9}{R0_VAL:>8.3f}")
for run, s in summaries.items():
    print(f"{run:<4}{s['name']:<26}{s['params'] / 1e6:>8.2f}M{s['val']:>8.3f}{s['train']:>8.3f}"
          f"{s['train'] - s['val']:>8.3f}{s['epoch']:>5d}/{s['epochs_run']:<3d}{s['hours']:>7.2f}")

winner = None
for run, s in summaries.items():    # RUNS order: simplest first
    if winner is None or s["val"] >= summaries[winner]["val"] + TIE_MARGIN:
        winner = run
print("\nWinner on validation:", winner, "| margin required:", TIE_MARGIN)
if QUICK_CHECK:
    print("Quick check: these numbers mean nothing.")

if len(histories) > 1:
    display(ShowImage(filename=str(plot_run_comparison(histories, OUTPUT_ROOT / f"run_comparison{SUFFIX}.png"))))
for s in summaries.values():
    print(s["name"])
    display(ShowImage(filename=str(s["dir"] / "curves.png")))

## 8. Test set and Grad-CAM for the winner

Only the winner is evaluated, using the CUB test split (5,794 images). The test score must not be used to choose
further experiments or settings, or it stops being a fair estimate. A quick check uses the validation split instead.

In [ ]:
from src.evaluation.evaluate import evaluate_checkpoint
from src.visualization.gradcam import run_gradcam

if RUN_TEST_EVAL and winner:
    checkpoint = summaries[winner]["dir"] / "best.pt"
    split = "val" if QUICK_CHECK else "test"

    report = evaluate_checkpoint(str(checkpoint), split)
    print(f"{winner} {split} accuracy: {report['accuracy']:.4f} | macro F1: {report['macro_f1']:.4f}")
    print("worst classes:", [row["class"] for row in report["worst_classes"][:5]])
    display(ShowImage(filename=str(checkpoint.parent / f"eval_{split}" / "confusion_matrix.png"), width=600))

    for kind, path in run_gradcam(str(checkpoint), split, n_correct=8, n_wrong=8).items():
        print(kind)
        display(ShowImage(filename=str(path)))
else:
    print("Skipped: no winner, or RUN_TEST_EVAL is off")

## 9. Save the results

In [ ]:
import shutil

if ON_KAGGLE or ON_COLAB:
    archive = shutil.make_archive(("/kaggle/working" if ON_KAGGLE else "/content") + "/resnet50d_results",
                                  "zip", OUTPUTS)
    print("results zipped to", archive, "- download it before the session ends")
else:
    print("results are in", OUTPUT_ROOT.resolve())
print("Settings to reuse next session:", json.dumps(tuned))
print(f"total time: {hours_used():.2f} h")